In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "***"


In [4]:
# !pip install -q langchain langchain-google-genai chromadb pypdf langchain-community langchain-text-splitters google-generativeai

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports successful!")

All imports successful!


In [7]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    ),
    Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    ),
    Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    ),
    Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    ),
    Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )
]

In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="my_chroma_db",
    collection_name="sample"
)

In [14]:
# add documents
vector_store.add_documents(documents)

['697edbec-b30b-4617-a08d-7b350ad1f8fd',
 'dca3331e-18d6-46ed-af10-dbae90e3a104',
 '32d20784-da76-45ad-90f0-081c87985c72',
 '6e3fee9c-f328-4a96-91e7-7b97c5e4d95f',
 'ea64c688-3433-41fe-a933-b1b7b09160f4']

In [15]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['697edbec-b30b-4617-a08d-7b350ad1f8fd',
  'dca3331e-18d6-46ed-af10-dbae90e3a104',
  '32d20784-da76-45ad-90f0-081c87985c72',
  '6e3fee9c-f328-4a96-91e7-7b97c5e4d95f',
  'ea64c688-3433-41fe-a933-b1b7b09160f4'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [16]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [17]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

In [19]:
results = vector_store.similarity_search_with_score(
    query="Who is the captain?",
    filter={"team": "Chennai Super Kings"}
)

for doc, score in results:
    print(score)
    print(doc.page_content)

0.6648716926574707
MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.
0.7477670907974243
Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.


In [20]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='697edbec-b30b-4617-a08d-7b350ad1f8fd', document=updated_doc1)


In [21]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['697edbec-b30b-4617-a08d-7b350ad1f8fd',
  'dca3331e-18d6-46ed-af10-dbae90e3a104',
  '32d20784-da76-45ad-90f0-081c87985c72',
  '6e3fee9c-f328-4a96-91e7-7b97c5e4d95f',
  'ea64c688-3433-41fe-a933-b1b7b09160f4'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a sin

In [22]:
# delete document
# vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [23]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['697edbec-b30b-4617-a08d-7b350ad1f8fd',
  'dca3331e-18d6-46ed-af10-dbae90e3a104',
  '32d20784-da76-45ad-90f0-081c87985c72',
  '6e3fee9c-f328-4a96-91e7-7b97c5e4d95f',
  'ea64c688-3433-41fe-a933-b1b7b09160f4'],
 'embeddings': array([[-0.00719311,  0.02029932,  0.02837158, ...,  0.01391   ,
         -0.01240376, -0.00345245],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a sin